### imports and definitions

In [ ]:
import numpy as np
from scipy.stats import norm, multivariate_normal
import matplotlib.pyplot as plt
import ipywidgets as widgets
%matplotlib widget

def add_play_button(ax, interval=50):
    def _spin(azim=0):
        ax.view_init(elev=ax.elev, azim=azim)
        ax.figure.canvas.draw_idle()
        
    play = widgets.Play(min=0, max=360, step=2, interval=interval)
    return widgets.interact(_spin, azim=play)

def plot_3D_scatter():
    fig = plt.figure(figsize=(6,6))
    ax = fig.add_subplot(projection="3d")
    ax.scatter(X[:, 0], X[:, 1], Y, color="k", s=10, alpha=1)
    x1_grid, x2_grid = np.meshgrid(
        np.linspace(X[:, 0].min(), X[:, 0].max(), 2),
        np.linspace(X[:, 1].min(), X[:, 1].max(), 2) )
    y_grid = Theta0 + Theta1[0] * x1_grid + Theta1[1] * x2_grid
    ax.plot_surface(x1_grid, x2_grid, y_grid, color="blue", alpha=0.4)
    ax.set_zlim(-50,50)
    ax.set_xlabel("X1")
    ax.set_ylabel("X2")
    ax.set_zlabel("Y")
    add_play_button(ax)
    return ax, x1_grid, x2_grid

def plot_3D_scatter_with_model(ThetaHat0,ThetaHat1):
    ax, x1_grid, x2_grid = plot_3D_scatter()
    yhat_grid = ThetaHat0 + ThetaHat1[0] * x1_grid + ThetaHat1[1] * x2_grid
    ax.plot_surface(x1_grid, x2_grid, yhat_grid, color="magenta", alpha=0.4)

def plot_gaussian(rv,sample,truevalue,subindex):

    gamma = 0.90 
    rho = rv.std()*abs(norm().ppf((1-gamma)/2))  

    _, ax = plt.subplots(figsize=(6,3),nrows=1)
    minx = rv.mean()-3*rv.std()
    minx = min((minx,-rv.std()))
    maxx = rv.mean()+3*rv.std()
    maxx = max((maxx,rv.std()))
    x = np.linspace(minx,maxx,200)
    y = rv.pdf(x)
    ind = (x>rv.mean()-rho) & (x<rv.mean()+rho)
    ax.fill_between(x[ind],y[ind],alpha=0.3)
    ax.plot(x,y)
    ax.axvline(truevalue,color='k',linestyle=':')
    ax.plot(sample,0,'o',color='m',markersize=10)
    ax.plot([sample-rho,sample+rho],[0,0],'-',color='m',linewidth=2)
    ax.plot([sample-rho,sample+rho],[0,0],'+',color='m',markersize=10,markeredgewidth=3)
    ax.spines[['left','top','right']].set_visible(False)
    ax.spines['bottom'].set_position('zero')
    ax.set_yticks([])
    ax.set_xticks([0,sample,truevalue])
    ax.set_xticklabels(['0',rf'$\hat\theta_{subindex}$',rf'$\theta_{subindex}$'],fontsize=16)

### Generate some input data $\mathbf{X}$ from a joint Gaussian distribution

In [ ]:
muX = (1,2)    # mean of the joint distribution

# variance of the joint distribution
varX1 = 10     # variance of X1
varX2 = 15     # variance of X2
covX1X2 = 3    # covariance between X1 and X3

# assemble into the covariance matrix
CovX = np.array([[varX1,covX1X2],[covX1X2,varX2]])

# use SciPy to generate the samples

N = 100 # number of samples
X = multivariate_normal(muX, CovX).rvs(size=N)  # sample of multivariate normal 

### Generate the corresponding output data $\mathbf{Y}$

In [ ]:
# the true input output map f(x) = theta0 + x theta1
Theta0 = ...
Theta1 = np.array([...,...])
Y = ...

# Y is corrupted by irreducible (aka measurement) noise
sigma2 = ...
Eps = ...

# scatter plot
plot_3D_scatter();

## Linear regression

In [ ]:
Xe = ...
ThetaHat = ...

# unpack
ThetaHat0 = ThetaHat[0]
ThetaHat1 = ThetaHat[1:]

# scatter plot
plot_3D_scatter_with_model(ThetaHat0,ThetaHat1)

### Confidence interval for the slope parameters $\underline{\theta}_1$

In [ ]:
varThetaHat1 = sigma2*np.linalg.inv(CovX)/(N-1)

### Confidence interval for $\theta_1$

In [ ]:
rvThetaHat1_0 = norm( loc=... , scale=... )

plot_gaussian(rv = rvThetaHat1_0,
              sample = ThetaHat1[0],
              truevalue = Theta1[0],
              subindex=1)

### Confidence interval for $\theta_2$

In [ ]:
rvThetaHat1_1 = norm( loc=... , scale=...)

plot_gaussian(rv = rvThetaHat1_1,
              sample = ThetaHat1[1],
              truevalue = Theta1[1], 
              subindex=2)